In [19]:
import pandas as pd

data = pd.read_csv('/scratch2/mlavechin/phorec/phonetically_transcribed_pairs/utterances2.csv', sep='\t')

/tmp/ipykernel_3323231/3868874667.py:3: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('/scratch2/mlavechin/phorec/phonetically_transcribed_pairs/utterances2.csv', sep='\t')


In [32]:
import pandas as pd
import numpy as np
from IPython.display import display, Audio, HTML, clear_output
import librosa
import random

class EasyAudioAnnotationInterface:
    def __init__(self, data, first_corpus=None, random_seed=42):
        self.data = data
        self.current_corpus_idx = 0
        self.current_file_idx = 0
        
        # Set random seed for reproducibility
        random.seed(random_seed)
        np.random.seed(random_seed)
        
        # Get sorted list of corpora (case-insensitive)
        self.corpora = sorted(data['corpus'].unique(), key=str.lower)
        
        # Set starting corpus
        if first_corpus and first_corpus in self.corpora:
            self.current_corpus_idx = self.corpora.index(first_corpus)
        
        # Initialize data structures
        self.current_corpus_files = []
        self.current_file_utterances = []
        
        # Load first corpus but don't display
        self.load_corpus_silent()
    
    def load_corpus_silent(self):
        """Load and sample files for current corpus without displaying"""
        if self.current_corpus_idx >= len(self.corpora):
            return
        
        current_corpus = self.corpora[self.current_corpus_idx]
        corpus_data = self.data[self.data['corpus'] == current_corpus]
        
        # Get unique audio files and sample 10% with minimum of 2 files
        unique_files = corpus_data['audio_path'].unique()
        n_sample = max(2, int(len(unique_files) * 0.1))  # Changed from max(1, ...) to max(2, ...)
        n_sample = min(n_sample, len(unique_files))  # Don't sample more files than available
        
        # Use random.sample for consistent seeding
        self.current_corpus_files = random.sample(list(unique_files), n_sample)
        
        self.current_file_idx = 0
        self.load_file_silent()
    
    def load_file_silent(self):
        """Load utterances for current file without displaying"""
        if self.current_file_idx >= len(self.current_corpus_files):
            # Move to next corpus
            self.current_corpus_idx += 1
            self.load_corpus_silent()
            return
        
        if self.current_file_idx < 0:
            # Can't go before first file
            self.current_file_idx = 0
            return
        
        current_file = self.current_corpus_files[self.current_file_idx]
        current_corpus = self.corpora[self.current_corpus_idx]
        
        # Get utterances for this file
        file_data = self.data[(self.data['corpus'] == current_corpus) & 
                             (self.data['audio_path'] == current_file)]
        
        # Sample 10 utterances (or all if less than 10)
        n_sample = min(10, len(file_data))
        self.current_file_utterances = file_data.sample(n=n_sample, random_state=42).reset_index(drop=True)
    
    def load_audio_segment(self, utterance):
        """Load and return audio segment"""
        try:
            audio_path = utterance['audio_path']
            onset_ms = utterance['onset']
            offset_ms = utterance['offset']
            
            # Convert milliseconds to seconds
            onset = onset_ms / 1000.0
            offset = offset_ms / 1000.0
            duration = offset - onset
            
            # Load only the relevant audio segment
            segment, sr = librosa.load(audio_path, sr=None, offset=onset, duration=duration)
            return Audio(data=segment, rate=sr, autoplay=False)
            
        except Exception as e:
            print(f"Error loading audio: {e}")
            return HTML("<p style='color: red;'>Error loading audio segment</p>")
    
    def show(self):
        """Display all utterances for current file"""
        if len(self.current_file_utterances) == 0:
            print("🎉 All corpora completed!")
            return
        
        # Display file info
        current_corpus = self.corpora[self.current_corpus_idx]
        filename = self.current_corpus_files[self.current_file_idx].split('/')[-1]
        
        print("="*80)
        print(f"Corpus: {current_corpus} ({self.current_corpus_idx + 1}/{len(self.corpora)})")
        print(f"Audio File: {filename} ({self.current_file_idx + 1}/{len(self.current_corpus_files)})")
        print(f"Utterances: {len(self.current_file_utterances)}")
        print("="*80)
        
        # Display all utterances
        for i, utterance in self.current_file_utterances.iterrows():
            print(f"\n--- Utterance {i + 1}/{len(self.current_file_utterances)} ---")
            
            # Show content preview
            content = utterance.get('content', 'N/A')
            print(f"Content: {content[:100]}{'...' if len(content) > 100 else ''}")
            
            # Display audio
            audio_widget = self.load_audio_segment(utterance)
            display(audio_widget)
            
            # Display phones
            phones = utterance.get('simplified_phones', 'N/A')
            phones_html = f"""
            <div style='font-family: monospace; font-size: 14px; padding: 10px; 
                       border: 2px solid #4CAF50; background-color: #f0f8ff; margin: 5px 0;'>
                <strong>Simplified Phones:</strong> <span style='color: #2E8B57;'>{phones}</span>
            </div>
            """
            display(HTML(phones_html))
        
        # Show navigation help
        print("\n" + "="*80)
        print("🎯 NAVIGATION:")
        print("   interface.next_file() ← Go to next audio file")
        print("   interface.prev_file() ← Go to previous audio file")
        print("   interface.show()      ← Refresh current display")
        print("="*80)
    
    def next_file(self):
        """Go to next file"""
        if self.current_file_idx < len(self.current_corpus_files) - 1:
            self.current_file_idx += 1
            self.load_file_silent()
            self.show()
        else:
            print("✅ All files in this corpus reviewed!")
            print("💡 Moving to next corpus...")
            self.current_corpus_idx += 1
            self.current_file_idx = 0
            self.load_corpus_silent()
            if self.current_corpus_idx < len(self.corpora):
                self.show()
    
    def prev_file(self):
        """Go to previous file"""
        if self.current_file_idx > 0:
            self.current_file_idx -= 1
            self.load_file_silent()
            self.show()
        else:
            print("Already at first file in corpus")
            if self.current_corpus_idx > 0:
                print("💡 Use interface.prev_corpus() to go to previous corpus")
    
    def prev_corpus(self):
        """Go to previous corpus"""
        if self.current_corpus_idx > 0:
            self.current_corpus_idx -= 1
            # Go to last file of previous corpus
            self.load_corpus_silent()
            self.current_file_idx = len(self.current_corpus_files) - 1
            self.load_file_silent()
            self.show()
        else:
            print("Already at first corpus")

    def play_utterance_by_index(self, index):
        """Play a specific utterance by its dataframe index"""
        try:
            utterance = self.data.iloc[index]
            audio_widget = self.load_audio_segment(utterance)
            
            # Show some info about the utterance
            print(f"Playing utterance {index}:")
            print(f"Corpus: {utterance.get('corpus', 'N/A')}")
            print(f"Content: {utterance.get('content', 'N/A')[:100]}...")
            print(f"Simplified phones: {utterance.get('simplified_phones', 'N/A')[:100]}...")
            print(f"Duration: {(utterance['offset'] - utterance['onset'])/1000:.2f}s")
            
            return audio_widget  # Jupyter will auto-display this
        except Exception as e:
            print(f"Error playing utterance {index}: {e}")


    def save_utterance_by_index(self, index, output_path=None):
        """Save a specific utterance by its dataframe index"""
        try:
            utterance = self.data.iloc[index]
            
            # Load audio segment
            audio_path = utterance['audio_path']
            onset_ms = utterance['onset']
            offset_ms = utterance['offset']
            
            onset = onset_ms / 1000.0
            offset = offset_ms / 1000.0
            duration = offset - onset
            
            # Load the audio segment
            segment, sr = librosa.load(audio_path, sr=None, offset=onset, duration=duration)
            
            # Generate output filename if not provided
            if output_path is None:
                corpus = utterance.get('corpus', 'unknown')
                filename = f"utterance_{index}_{corpus}_{onset:.2f}s-{offset:.2f}s.wav"
                output_path = filename
            
            # Save the audio
            import soundfile as sf
            sf.write(output_path, segment, sr)
            
            print(f"Saved utterance {index} to: {output_path}")
            print(f"Duration: {duration:.2f}s")
            print(f"Sample rate: {sr} Hz")
            
            return output_path
        
        except Exception as e:
            print(f"Error saving utterance {index}: {e}")

def create_easy_annotation_interface(data, first_corpus=None, random_seed=42):
    """
    Create an easy-to-use annotation interface that works reliably in Jupyter
    """
    return EasyAudioAnnotationInterface(data, first_corpus, random_seed)

# Usage:
interface = create_easy_annotation_interface(data, first_corpus="PhonoDis", random_seed=42)
# # Nothing is displayed initially
# interface.next_file()  # Go to next audio file  
# interface.prev_file()  # Go to previous audio file
# interface.show()       # Refresh display

In [8]:
interface.next_file()

Corpus: PhonoDis (20/31)
Audio File: ChildO.wav (2/2)
Utterances: 10

--- Utterance 1/10 ---
Content: castanho .



--- Utterance 2/10 ---
Content: zero .



--- Utterance 3/10 ---
Content: sede .



--- Utterance 4/10 ---
Content: lã .



--- Utterance 5/10 ---
Content: gruta .


...


--- Utterance 7/10 ---
Content: tigres .



--- Utterance 8/10 ---
Content: bruxa .



--- Utterance 9/10 ---
Content: cobra .



--- Utterance 10/10 ---
Content: guitarra .



🎯 NAVIGATION:
   interface.next_file() ← Go to next audio file
   interface.prev_file() ← Go to previous audio file
   interface.show()      ← Refresh current display


In [39]:
# File-level annotation
class SingleFileReviewInterface:
    
    def __init__(self, data, corpus_name, random_seed=42):
        self.data = data
        self.corpus_name = corpus_name
        self.current_file_idx = 0
        
        # Set random seed for reproducibility
        random.seed(random_seed)
        np.random.seed(random_seed)
        
        # Get all files for this corpus
        corpus_data = self.data[self.data['corpus'] == corpus_name]
        unique_files = corpus_data['audio_path'].unique()
        
        # Use all files for detailed review and sort alphabetically
        self.corpus_files = sorted(unique_files)  # Changed from list(unique_files)
        
        # Load first file but don't display
        self.load_file_silent()
    
    def load_file_silent(self):
        """Load utterances for current file without displaying"""
        if self.current_file_idx >= len(self.corpus_files):
            print("🎉 All files in corpus completed!")
            return
        
        if self.current_file_idx < 0:
            self.current_file_idx = 0
            return
        
        current_file = self.corpus_files[self.current_file_idx]
        
        # Get utterances for this file
        file_data = self.data[
            (self.data['corpus'] == self.corpus_name) & 
            (self.data['audio_path'] == current_file)
        ]
        
        # Sample 10 utterances (or all if less than 10)
        n_sample = min(10, len(file_data))
        self.current_file_utterances = file_data.sample(n=n_sample, random_state=42).reset_index(drop=True)
    
    def load_audio_segment(self, utterance):
        """Load and return audio segment"""
        try:
            audio_path = utterance['audio_path']
            onset_ms = utterance['onset']
            offset_ms = utterance['offset']
            
            # Convert milliseconds to seconds
            onset = onset_ms / 1000.0
            offset = offset_ms / 1000.0
            duration = offset - onset
            
            # Load only the relevant audio segment
            segment, sr = librosa.load(audio_path, sr=None, offset=onset, duration=duration)
            return Audio(data=segment, rate=sr, autoplay=False)
            
        except Exception as e:
            print(f"Error loading audio: {e}")
            return HTML("<p style='color: red;'>Error loading audio segment</p>")
    
    def show(self):
        """Display current file with ALL its utterances"""
        if len(self.current_file_utterances) == 0:
            print("🎉 All files completed!")
            return
        
        # Display file info
        filename = self.corpus_files[self.current_file_idx].split('/')[-1]
        
        print("="*80)
        print(f"📁 SINGLE FILE REVIEW")
        print(f"Corpus: {self.corpus_name}")
        print(f"Audio File: {filename} ({self.current_file_idx + 1}/{len(self.corpus_files)})")
        print(f"Utterances: {len(self.current_file_utterances)} (sampled)")
        print("="*80)
        
        # Display all utterances for this file
        for i, utterance in self.current_file_utterances.iterrows():
            print(f"\n--- Utterance {i + 1}/{len(self.current_file_utterances)} ---")
            
            # Show content preview
            content = utterance.get('content', 'N/A')
            print(f"Content: {content[:100]}{'...' if len(content) > 100 else ''}")
            
            # Display audio
            audio_widget = self.load_audio_segment(utterance)
            display(audio_widget)
            
            # Display phones
            phones = utterance.get('simplified_phones', 'N/A')
            phones_html = f"""
            <div style='font-family: monospace; font-size: 14px; padding: 10px; 
                       border: 2px solid #FF6B6B; background-color: #fff5f5; margin: 5px 0;'>
                <strong>Simplified Phones:</strong> <span style='color: #D63384;'>{phones}</span>
            </div>
            """
            display(HTML(phones_html))
        
        # Show navigation help
        print("\n" + "="*80)
        print("🎯 NAVIGATION:")
        print("   single_review.next_file() ← Go to next audio file")
        print("   single_review.prev_file() ← Go to previous audio file")
        print("   single_review.show()      ← Refresh current display")
        print("   single_review.summary()   ← Show corpus summary")
        print("="*80)
    
    def next_file(self):
        """Go to next file"""
        if self.current_file_idx < len(self.corpus_files) - 1:
            self.current_file_idx += 1
            self.load_file_silent()
            self.show()
        else:
            print("✅ All files in this corpus reviewed!")
            print("🎉 Corpus annotation complete!")
    
    def prev_file(self):
        """Go to previous file"""
        if self.current_file_idx > 0:
            self.current_file_idx -= 1
            self.load_file_silent()
            self.show()
        else:
            print("Already at first file in corpus")
    
    def jump_to_file(self, file_number):
        """Jump to specific file number (1-indexed)"""
        if 1 <= file_number <= len(self.corpus_files):
            self.current_file_idx = file_number - 1
            self.load_file_silent()
            self.show()
        else:
            print(f"File number must be between 1 and {len(self.corpus_files)}")
    
    def summary(self):
        """Show corpus summary"""
        print("="*60)
        print(f"📊 CORPUS SUMMARY: {self.corpus_name}")
        print(f"Total Files: {len(self.corpus_files)}")
        print(f"Current File: {self.current_file_idx + 1}/{len(self.corpus_files)}")
        
        # Show file list with utterance counts
        corpus_data = self.data[self.data['corpus'] == self.corpus_name]
        print("\nFiles in corpus (total utterances, 10 sampled for display):")
        for i, file_path in enumerate(self.corpus_files):
            filename = file_path.split('/')[-1]
            utterance_count = len(corpus_data[corpus_data['audio_path'] == file_path])
            marker = "👉" if i == self.current_file_idx else "  "
            print(f"{marker} {i+1:2d}. {filename} ({utterance_count} utterances)")
        print("="*60)

def create_single_file_review(data, corpus_name, random_seed=42):
    """
    Create a single-file review interface for detailed corpus annotation
    Shows 10 randomly sampled utterances per file (same as batch interface)
    """
    available_corpora = sorted(data['corpus'].unique(), key=str.lower)
    if corpus_name not in available_corpora:
        print(f"❌ Corpus '{corpus_name}' not found!")
        print(f"Available corpora: {available_corpora}")
        return None
    
    return SingleFileReviewInterface(data, corpus_name, random_seed)

# USAGE EXAMPLES:

# Example 1: Review all files in PhonoDis corpus one by one
single_review = create_single_file_review(data, "PhonBLA", random_seed=42)

# Example 2: Navigation
# single_review.next_file()     # Go to first file (shows 10 sampled utterances)
# single_review.next_file()     # Go to second file  
# single_review.prev_file()     # Go back to first file
# single_review.jump_to_file(5) # Jump directly to file #5
# single_review.summary()       # Show corpus overview

# Example 3: Review different corpus
# single_review_2 = create_single_file_review(data, "McAllister")
# single_review_2.next_file()   # Start reviewing McAllister corpus

In [46]:
single_review.next_file()

📁 SINGLE FILE REVIEW
Corpus: PhonBLA
Audio File: 030904.wav (6/268)
Utterances: 10 (sampled)

--- Utterance 1/10 ---
Content: Ja .



--- Utterance 2/10 ---
Content: Dunkel .



--- Utterance 3/10 ---
Content: Ja .



--- Utterance 4/10 ---
Content: Ich 0sehe 0was , was Du nicht siehst und das (..) yyy ist yyy gelb .



--- Utterance 5/10 ---
Content: Das Schiff .


...


--- Utterance 7/10 ---
Content: Ja .



--- Utterance 8/10 ---
Content: Spielen .



--- Utterance 9/10 ---
Content: Der Toaster .



--- Utterance 10/10 ---
Content: Nein .



🎯 NAVIGATION:
   single_review.next_file() ← Go to next audio file
   single_review.prev_file() ← Go to previous audio file
   single_review.show()      ← Refresh current display
   single_review.summary()   ← Show corpus summary


In [33]:
interface.save_utterance_by_index(525682, "/scratch2/mlavechin/phorec/analysis/examples/30_sec.wav")

Saved utterance 525682 to: /scratch2/mlavechin/phorec/analysis/examples/30_sec.wav
Duration: 29.99s
Sample rate: 16000 Hz


'/scratch2/mlavechin/phorec/analysis/examples/30_sec.wav'

In [28]:
data.loc[272927]['phones']

'kojøjø tẽja'